# 02 — Data Quality Audit

**IT3051 – Fundamentals of Data Mining · Mini Project 2026**

This audit asks: **What quality issues exist in the raw data, and which require decisions during preprocessing?** It systematically investigates missing data, exact duplicate records, invalid or unusual values, category consistency, numerical ranges, and possible outlier candidates in the Hotel Booking Demand dataset.

Detection of a data-quality issue does not automatically justify removing or modifying it. Each treatment decision will be made later during preprocessing. **The raw dataset must remain unchanged.** No cleaning, encoding, modelling, leakage removal, or visual EDA is performed.

## 2 — Imports and data loading

Use the same project-root check as the data-understanding notebook. Execution is supported from the repository root or `notebooks`. Load the CSV using pandas' default parser; missing-value findings therefore describe pandas-parsed values. Text labels that pandas interprets as missing will appear in the missing-value audit rather than the category audit.

A SHA-256 fingerprint records the raw file's contents for the final integrity check. All percentages use all dataset rows unless a table explicitly states a different denominator.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "requirements.txt").is_file()
     and (candidate / "notebooks").is_dir()
     and (candidate / "data" / "raw").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")
data_path = project_root / "data" / "raw" / "hotel_bookings.csv"
if not data_path.is_file():
    raise FileNotFoundError("Place hotel_bookings.csv inside data/raw/ before running this notebook.")

raw_hash_before = hashlib.sha256(data_path.read_bytes()).hexdigest()
df = pd.read_csv(data_path)
original_shape = df.shape
original_columns = df.columns.tolist()
print(f"Loaded {data_path.relative_to(project_root).as_posix()}: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"pandas version: {pd.__version__}")

def percent(count, denominator):
    return 100 * count / denominator if denominator else float("nan")

Loaded data/raw/hotel_bookings.csv: 119,390 rows × 32 columns
pandas version: 3.0.3


## 3 — Raw data integrity check

List every column and confirm the target and hotel grouping field exist. Report unexpected non-null target values and hotel categories without correcting them. Missing values are reported separately. Expected target labels are **0 = Not Cancelled** and **1 = Cancelled**; expected hotel categories are **City Hotel** and **Resort Hotel**.

In [2]:
required = {"is_canceled", "hotel"}
missing_required = sorted(required - set(df.columns))
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")
display(pd.DataFrame({"Number": range(1, df.shape[1] + 1), "Column": df.columns}))
print(f"Row count: {len(df):,}; column count: {len(df.columns)}")
unexpected_target = df["is_canceled"].notna() & ~df["is_canceled"].isin([0, 1])
unexpected_hotel = df["hotel"].notna() & ~df["hotel"].isin(["City Hotel", "Resort Hotel"])
integrity_summary = pd.DataFrame([
    {"Field": "is_canceled", "Observed non-null values": df.loc[df["is_canceled"].notna(), "is_canceled"].unique().tolist(),
     "Unexpected rows": int(unexpected_target.sum()), "Missing rows": int(df["is_canceled"].isna().sum())},
    {"Field": "hotel", "Observed non-null values": df.loc[df["hotel"].notna(), "hotel"].unique().tolist(),
     "Unexpected rows": int(unexpected_hotel.sum()), "Missing rows": int(df["hotel"].isna().sum())},
])
display(integrity_summary)
absent_hotels = sorted({"City Hotel", "Resort Hotel"} - set(df["hotel"].unique()))
print("Expected hotel categories absent:", absent_hotels)
if unexpected_target.any():
    display(df.loc[unexpected_target, "is_canceled"].value_counts().head(10))
if unexpected_hotel.any():
    display(df.loc[unexpected_hotel, "hotel"].value_counts().head(10))

,Number,Column
0,1,hotel
1,2,is_canceled
2,3,lead_time
3,4,arrival_date_year
4,5,arrival_date_month
5,6,arrival_date_week_number
6,7,arrival_date_day_of_month
7,8,stays_in_weekend_nights
8,9,stays_in_week_nights
9,10,adults


Row count: 119,390; column count: 32


,Field,Observed non-null values,Unexpected rows,Missing rows
0,is_canceled,"[0, 1]",0,0
1,hotel,"[Resort Hotel, City Hotel]",0,0


Expected hotel categories absent: []


## 4 — Missing-value audit

Show all columns ordered by missing percentage, then isolate affected columns. Missingness is an observation, not proof that a record is incorrect. The hotel comparison uses each hotel's own record count as its denominator; it describes differences without attributing causes.

In [3]:
missing_summary = pd.DataFrame({
    "Feature": df.columns,
    "Missing count": df.isna().sum().to_numpy(),
    "Missing percentage": (df.isna().mean() * 100).to_numpy(),
    "dtype": [str(dtype) for dtype in df.dtypes],
}).sort_values(["Missing percentage", "Feature"], ascending=[False, True], ignore_index=True)
display(missing_summary.round(3))
affected_missing = missing_summary.loc[missing_summary["Missing count"] > 0]
print(f"Columns containing missing values: {len(affected_missing)}")
display(affected_missing.round(3))

missing_hotel_rows = []
for hotel, group in df.groupby("hotel", dropna=False, sort=True):
    for column in affected_missing["Feature"]:
        count = int(group[column].isna().sum())
        missing_hotel_rows.append({"Hotel": hotel, "Feature": column,
            "Hotel records": len(group), "Missing count": count,
            "Missing percentage within hotel": percent(count, len(group))})
missing_by_hotel = pd.DataFrame(missing_hotel_rows)
display(missing_by_hotel.round(3))

,Feature,Missing count,Missing percentage,dtype
0,company,112593,94.307,float64
1,agent,16340,13.686,float64
2,country,488,0.409,str
3,children,4,0.003,float64
4,adr,0,0.000,float64
5,adults,0,0.000,int64
6,arrival_date_day_of_month,0,0.000,int64
7,arrival_date_month,0,0.000,str
8,arrival_date_week_number,0,0.000,int64
9,arrival_date_year,0,0.000,int64


Columns containing missing values: 4


,Feature,Missing count,Missing percentage,dtype
0,company,112593,94.307,float64
1,agent,16340,13.686,float64
2,country,488,0.409,str
3,children,4,0.003,float64


,Hotel,Feature,Hotel records,Missing count,Missing percentage within hotel
0,City Hotel,company,79330,75641,95.350
1,City Hotel,agent,79330,8131,10.250
2,City Hotel,country,79330,24,0.030
3,City Hotel,children,79330,4,0.005
4,Resort Hotel,company,40060,36952,92.242
5,Resort Hotel,agent,40060,8209,20.492
6,Resort Hotel,country,40060,464,1.158
7,Resort Hotel,children,40060,0,0.000


## 5 — Exact duplicate audit

Equality is evaluated across **all columns**, including the target, on the parsed DataFrame. Matching missing values are included. Distinguish:

- **Extra identical copies:** all but the first row in each identical-row group (`keep="first"`).
- **Rows involved in repeated groups:** every row in groups with at least two identical records (`keep=False`).
- **Distinct full-row patterns:** the number of unique complete-row combinations, including patterns occurring once.

The dataset does not contain an obvious unique booking identifier, so identical records cannot automatically be assumed to represent accidental duplicate bookings. Duplicate removal requires justification during preprocessing; none are removed here.

In [4]:
duplicate_extra = df.duplicated(keep="first")
duplicate_involved = df.duplicated(keep=False)
row_frequencies = df.value_counts(dropna=False)
repeated_groups = row_frequencies.loc[row_frequencies > 1]
duplicate_summary = pd.DataFrame([
    {"Measure": "Extra identical copies (beyond first)", "Count": int(duplicate_extra.sum()),
     "Percentage of all rows": percent(duplicate_extra.sum(), len(df))},
    {"Measure": "All rows involved in repeated groups", "Count": int(duplicate_involved.sum()),
     "Percentage of all rows": percent(duplicate_involved.sum(), len(df))},
    {"Measure": "Distinct full-row patterns", "Count": len(row_frequencies), "Percentage of all rows": pd.NA},
    {"Measure": "Repeated full-row groups", "Count": len(repeated_groups), "Percentage of all rows": pd.NA},
    {"Measure": "Maximum identical-row frequency", "Count": int(row_frequencies.max()) if len(row_frequencies) else 0,
     "Percentage of all rows": pd.NA},
])
display(duplicate_summary)
print("Sample of at most five repeated full-row groups, ordered by frequency:")
with pd.option_context("display.max_columns", None):
    display(repeated_groups.head(5).rename("Frequency").reset_index())

,Measure,Count,Percentage of all rows
0,Extra identical copies (beyond first),31994,26.797889
1,All rows involved in repeated groups,40165,33.641846
2,Distinct full-row patterns,87396,<NA>
3,Repeated full-row groups,8171,<NA>
4,Maximum identical-row frequency,180,<NA>


Sample of at most five repeated full-row groups, ordered by frequency:


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,Frequency
0,City Hotel,1,277,2016,November,46,7,1,2,2,0.0,0,BB,PRT,Groups,TA/TO,0,0,0,A,A,0,Non Refund,NaN,NaN,0,Transient,100.0,0,0,Canceled,2016-04-04,180
1,City Hotel,1,68,2016,February,8,17,0,2,2,0.0,0,BB,PRT,Groups,TA/TO,0,1,0,A,A,0,Non Refund,37.0,NaN,0,Transient,75.0,0,0,Canceled,2016-01-06,150
2,City Hotel,1,188,2016,June,25,15,0,2,1,0.0,0,BB,PRT,Offline TA/TO,TA/TO,0,0,0,A,A,0,Non Refund,119.0,NaN,39,Transient,130.0,0,0,Canceled,2016-01-18,109
3,City Hotel,1,158,2016,May,22,24,0,2,1,0.0,0,BB,PRT,Groups,TA/TO,0,0,0,A,A,0,Non Refund,37.0,NaN,31,Transient,130.0,0,0,Canceled,2016-01-18,101
4,City Hotel,1,34,2015,December,50,8,0,2,1,0.0,0,BB,PRT,Offline TA/TO,TA/TO,0,1,0,A,A,0,Non Refund,19.0,NaN,0,Transient,90.0,0,0,Canceled,2015-11-17,100


## 6 — Basic validity checks

Check negative guest, stay, history, lead-time, and operational counts, along with structural date ranges and arrival-month labels. Bounds are screening rules: days 1–31 and weeks 1–53 do not establish whether a complete calendar date is valid. Missing values are counted separately and are not silently passed as validated observations. No values are corrected.

In [5]:
count_fields = [
    "adults", "children", "babies", "stays_in_weekend_nights", "stays_in_week_nights",
    "lead_time", "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "days_in_waiting_list", "required_car_parking_spaces", "total_of_special_requests",
]
audit_required = count_fields + ["arrival_date_day_of_month", "arrival_date_week_number", "arrival_date_month", "adr"]
absent_audit_columns = sorted(set(audit_required) - set(df.columns))
if absent_audit_columns:
    raise ValueError(f"Columns required by this audit are missing: {absent_audit_columns}")

validity_checks = [("Unexpected binary target value", "is_canceled", unexpected_target)]
for column in count_fields:
    validity_checks.append(("Negative count / duration", column, df[column].notna() & (df[column] < 0)))
validity_checks.extend([
    ("Day outside 1–31", "arrival_date_day_of_month", df["arrival_date_day_of_month"].notna() & ~df["arrival_date_day_of_month"].between(1, 31)),
    ("Week outside 1–53", "arrival_date_week_number", df["arrival_date_week_number"].notna() & ~df["arrival_date_week_number"].between(1, 53)),
])
month_labels = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
validity_checks.append(("Unexpected month label", "arrival_date_month", df["arrival_date_month"].notna() & ~df["arrival_date_month"].isin(month_labels)))
validity_summary = pd.DataFrame([
    {"Check": name, "Feature": column, "Flagged rows": int(mask.sum()),
     "Flagged percentage": percent(mask.sum(), len(df)), "Missing / not evaluated": int(df[column].isna().sum())}
    for name, column, mask in validity_checks
])
display(validity_summary.round(3))
for name, column, mask in validity_checks:
    if mask.any():
        print(f"Sample: {name}, {column}")
        display(df.loc[mask, ["hotel", column]].head(3))

,Check,Feature,Flagged rows,Flagged percentage,Missing / not evaluated
0,Unexpected binary target value,is_canceled,0,0.0,0
1,Negative count / duration,adults,0,0.0,0
2,Negative count / duration,children,0,0.0,4
3,Negative count / duration,babies,0,0.0,0
4,Negative count / duration,stays_in_weekend_nights,0,0.0,0
5,Negative count / duration,stays_in_week_nights,0,0.0,0
6,Negative count / duration,lead_time,0,0.0,0
7,Negative count / duration,previous_cancellations,0,0.0,0
8,Negative count / duration,previous_bookings_not_canceled,0,0.0,0
9,Negative count / duration,booking_changes,0,0.0,0


## 7 — Zero-guest and zero-stay investigation

Calculate temporary Series without adding columns to `df`. Direct addition preserves missingness: a booking with an unknown guest count is not assumed to have zero guests. Report unevaluable totals separately. Zero totals need contextual investigation before preprocessing; they are not automatically discarded.

In [6]:
total_guests = df["adults"] + df["children"] + df["babies"]
total_stay_nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
zero_guests = total_guests == 0
zero_stay = total_stay_nights == 0
zero_summary = pd.DataFrame([
    {"Check": "Zero total guests", "Rows": int(zero_guests.sum()), "Percentage of all rows": percent(zero_guests.sum(), len(df)),
     "Unevaluable totals": int(total_guests.isna().sum())},
    {"Check": "Zero total stay nights", "Rows": int(zero_stay.sum()), "Percentage of all rows": percent(zero_stay.sum(), len(df)),
     "Unevaluable totals": int(total_stay_nights.isna().sum())},
])
display(zero_summary.round(3))
sample_columns = ["hotel", "is_canceled", "adults", "children", "babies", "stays_in_weekend_nights", "stays_in_week_nights", "adr"]
print("Zero-guest sample (up to five records):")
display(df.loc[zero_guests, sample_columns].head(5))
print("Zero-stay sample (up to five records):")
display(df.loc[zero_stay, sample_columns].head(5))

,Check,Rows,Percentage of all rows,Unevaluable totals
0,Zero total guests,180,0.151,4
1,Zero total stay nights,715,0.599,0


Zero-guest sample (up to five records):


,hotel,is_canceled,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr
2224,Resort Hotel,0,0,0.0,0,0,3,0.0
2409,Resort Hotel,0,0,0.0,0,0,0,0.0
3181,Resort Hotel,0,0,0.0,0,1,2,0.0
3684,Resort Hotel,0,0,0.0,0,1,4,0.0
3708,Resort Hotel,0,0,0.0,0,2,4,0.0


Zero-stay sample (up to five records):


,hotel,is_canceled,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr
0,Resort Hotel,0,2,0.0,0,0,0,0.0
1,Resort Hotel,0,2,0.0,0,0,0,0.0
167,Resort Hotel,0,2,0.0,0,0,0,0.0
168,Resort Hotel,0,1,0.0,0,0,0,0.0
196,Resort Hotel,0,2,0.0,0,0,0,0.0


## 8 — ADR investigation

Inspect the recorded `adr` range, centre, selected quantiles, and negative/zero counts. Negative, zero, and unusually high values require contextual investigation; numerical extremity alone does not establish an error. ADR is not capped, replaced, or transformed.

In [7]:
adr_summary = df["adr"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("ADR")
display(adr_summary.round(3))
negative_adr = df["adr"] < 0
zero_adr = df["adr"] == 0
display(pd.DataFrame({"Observation": ["Negative ADR", "Zero ADR"],
    "Rows": [int(negative_adr.sum()), int(zero_adr.sum())]}))
print("Negative ADR sample:")
display(df.loc[negative_adr, sample_columns].head(3))
print("Three largest observed ADR values (screening sample):")
display(df.nlargest(3, "adr")[sample_columns])

,ADR
count,119390.000
mean,101.831
std,50.536
min,-6.380
1%,0.000
5%,38.400
25%,69.290
50%,94.575
75%,126.000
95%,193.500


,Observation,Rows
0,Negative ADR,1
1,Zero ADR,1959


Negative ADR sample:


,hotel,is_canceled,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr
14969,Resort Hotel,0,2,0.0,0,4,6,-6.38


Three largest observed ADR values (screening sample):


,hotel,is_canceled,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr
48515,City Hotel,1,2,0.0,0,0,1,5400.0
111403,City Hotel,0,1,0.0,0,0,1,510.0
15083,Resort Hotel,0,2,0.0,0,0,1,508.0


## 9 — Numerical range audit

Summarise meaningful numeric/count measurements. Numeric identifiers `agent` and `company`, target labels, binary category codes, and date components are excluded from continuous-range interpretation. Missing values do not contribute to numerical quantiles or means.

In [8]:
screening_columns = ["lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "days_in_waiting_list", "required_car_parking_spaces", "total_of_special_requests"]
range_summary = df[screening_columns].describe().T[["min", "25%", "50%", "75%", "max", "mean"]]
display(range_summary.rename(columns={"25%": "Q1", "50%": "Median", "75%": "Q3"}).round(3))

,min,Q1,Median,Q3,max,mean
lead_time,0.00,18.00,69.000,160.0,737.0,104.011
adr,-6.38,69.29,94.575,126.0,5400.0,101.831
stays_in_weekend_nights,0.00,0.00,1.000,2.0,19.0,0.928
stays_in_week_nights,0.00,1.00,2.000,3.0,50.0,2.500
adults,0.00,2.00,2.000,2.0,55.0,1.856
children,0.00,0.00,0.000,0.0,10.0,0.104
babies,0.00,0.00,0.000,0.0,10.0,0.008
previous_cancellations,0.00,0.00,0.000,0.0,26.0,0.087
previous_bookings_not_canceled,0.00,0.00,0.000,0.0,72.0,0.137
booking_changes,0.00,0.00,0.000,0.0,21.0,0.221


## 10 — Outlier candidate audit

Use the **1.5 × IQR rule as screening only**: bounds are Q1 − 1.5 × IQR and Q3 + 1.5 × IQR. Percentages here use the number of **non-null observations in each feature**. Counts overlap across features and must not be added to obtain a record total.

An **“IQR outlier” does not mean “invalid record.”** Discrete, sparse counts can have IQR = 0, so even ordinary nonzero values may be flagged. Context and later EDA must inform any treatment. No target labels, identifiers, categorical codes, or date components are screened.

In [9]:
iqr_rows = []
iqr_masks = {}
for column in screening_columns:
    values = df[column]
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = values.notna() & ((values < lower) | (values > upper))
    iqr_masks[column] = mask
    iqr_rows.append({"Feature": column, "Q1": q1, "Q3": q3, "IQR": iqr,
        "Lower bound": lower, "Upper bound": upper, "Non-null observations": int(values.count()),
        "Outside bounds": int(mask.sum()), "Percentage of non-null": percent(mask.sum(), values.count())})
iqr_summary = pd.DataFrame(iqr_rows)
display(iqr_summary.round(3))
iqr_any = pd.DataFrame(iqr_masks).any(axis=1)
print(f"Distinct records flagged by at least one screened feature: {int(iqr_any.sum()):,} ({percent(iqr_any.sum(), len(df)):.3f}% of all rows)")

,Feature,Q1,Q3,IQR,Lower bound,Upper bound,Non-null observations,Outside bounds,Percentage of non-null
0,lead_time,18.00,160.0,142.00,-195.000,373.000,119390,3005,2.517
1,adr,69.29,126.0,56.71,-15.775,211.065,119390,3793,3.177
2,stays_in_weekend_nights,0.00,2.0,2.00,-3.000,5.000,119390,265,0.222
3,stays_in_week_nights,1.00,3.0,2.00,-2.000,6.000,119390,3354,2.809
4,adults,2.00,2.0,0.00,2.000,2.000,119390,29710,24.885
5,children,0.00,0.0,0.00,0.000,0.000,119386,8590,7.195
6,babies,0.00,0.0,0.00,0.000,0.000,119390,917,0.768
7,previous_cancellations,0.00,0.0,0.00,0.000,0.000,119390,6484,5.431
8,previous_bookings_not_canceled,0.00,0.0,0.00,0.000,0.000,119390,3620,3.032
9,booking_changes,0.00,0.0,0.00,0.000,0.000,119390,18076,15.140


Distinct records flagged by at least one screened feature: 64,004 (53.609% of all rows)


## 11 — Categorical consistency audit

Inspect object/string columns for unique non-null values, literal empty strings, whitespace-only values, leading/trailing whitespace, and case-only variants. Temporary string operations are used for detection only. Case groups compare `casefold()` values without changing stored categories.

Search a stated, limited set of labels (`Undefined`, `Unknown`, `Unspecified`, `Not specified`, `N/A`, `NA`, `None`, `Null`, `Missing`) as contextual flags. Such labels are not declared invalid. Pandas may already have parsed some tokens or empty CSV fields as missing. Examples are limited to five per column; high-cardinality lists are not dumped.

In [10]:
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
marker_labels = {"undefined", "unknown", "unspecified", "not specified", "n/a", "na", "none", "null", "missing"}
category_rows = []
category_details = []
for column in text_columns:
    values = df[column]
    present = values.loc[values.notna()]
    stripped = present.str.strip()
    blank = present.eq("")
    whitespace_only = present.ne("") & stripped.eq("")
    padded = present.ne(stripped)
    variants = {}
    for value in present.unique():
        variants.setdefault(value.casefold(), []).append(value)
    case_groups = [group for group in variants.values() if len(group) > 1]
    case_values = [value for group in case_groups for value in group]
    marker_mask = stripped.str.casefold().isin(marker_labels)
    category_rows.append({"Feature": column, "Unique non-null": present.nunique(),
        "Empty strings": int(blank.sum()), "Whitespace-only": int(whitespace_only.sum()),
        "Leading/trailing whitespace": int(padded.sum()), "Case-variant groups": len(case_groups),
        "Rows in case-variant groups": int(present.isin(case_values).sum()),
        "Context-label rows": int(marker_mask.sum())})
    for label, count in present.loc[marker_mask].value_counts().head(5).items():
        category_details.append({"Feature": column, "Observation": "Context label", "Example": repr(label), "Rows": int(count)})
    for label, count in present.loc[padded].value_counts().head(5).items():
        category_details.append({"Feature": column, "Observation": "Whitespace", "Example": repr(label), "Rows": int(count)})
    for group in case_groups[:5]:
        category_details.append({"Feature": column, "Observation": "Case variants", "Example": repr(group), "Rows": int(present.isin(group).sum())})
category_summary = pd.DataFrame(category_rows)
category_examples = pd.DataFrame(category_details, columns=["Feature", "Observation", "Example", "Rows"])
display(category_summary)
display(category_examples)

,Feature,Unique non-null,Empty strings,Whitespace-only,Leading/trailing whitespace,Case-variant groups,Rows in case-variant groups,Context-label rows
0,hotel,2,0,0,0,0,0,0
1,arrival_date_month,12,0,0,0,0,0,0
2,meal,5,0,0,0,0,0,1169
3,country,177,0,0,0,0,0,0
4,market_segment,8,0,0,0,0,0,2
5,distribution_channel,5,0,0,0,0,0,5
6,reserved_room_type,10,0,0,0,0,0,0
7,assigned_room_type,12,0,0,0,0,0,0
8,deposit_type,3,0,0,0,0,0,0
9,customer_type,4,0,0,0,0,0,0


,Feature,Observation,Example,Rows
0,meal,Context label,'Undefined',1169
1,market_segment,Context label,'Undefined',2
2,distribution_channel,Context label,'Undefined',5


## 12 — Hotel-type data-quality comparison

For each hotel, show counts and percentages using its own number of records. “Duplicate copies” counts occurrences beyond the first full-row match; “duplicate involvement” includes every member of repeated groups. Since `hotel` is part of the complete row, matches cannot cross hotel categories. “Any missing value” counts records, not missing cells. Feature-specific missingness is shown in Section 4.

These are descriptive comparisons. They do not establish that one hotel has better or worse data, explain causes, or show that separate models will perform better.

In [11]:
hotel_rows = []
hotel_masks = {"Any missing value": df.isna().any(axis=1), "Duplicate copies": duplicate_extra,
    "Duplicate involvement": duplicate_involved, "Zero guests": zero_guests, "Zero stay": zero_stay,
    "Negative ADR": negative_adr, "Zero ADR": zero_adr}
for hotel, group in df.groupby("hotel", dropna=False, sort=True):
    for measure, mask in hotel_masks.items():
        count = int(mask.loc[group.index].sum())
        hotel_rows.append({"Hotel": hotel, "Hotel records": len(group), "Measure": measure,
            "Rows": count, "Percentage within hotel": percent(count, len(group))})
hotel_comparison = pd.DataFrame(hotel_rows)
display(hotel_comparison.round(3))

,Hotel,Hotel records,Measure,Rows,Percentage within hotel
0,City Hotel,79330,Any missing value,79283,99.941
1,City Hotel,79330,Duplicate copies,25902,32.651
2,City Hotel,79330,Duplicate involvement,31748,40.020
3,City Hotel,79330,Zero guests,167,0.211
4,City Hotel,79330,Zero stay,331,0.417
5,City Hotel,79330,Negative ADR,0,0.000
6,City Hotel,79330,Zero ADR,1208,1.523
7,Resort Hotel,40060,Any missing value,39890,99.576
8,Resort Hotel,40060,Duplicate copies,6092,15.207
9,Resort Hotel,40060,Duplicate involvement,8417,21.011


## 13 — Data-quality issue register

Include only issues detected by the calculations above. These entries identify questions to resolve, not approved treatments. A record can appear under multiple issues. Potential leakage is reserved for a dedicated audit.

In [12]:
issue_rows = []
def register(issue, features, evidence, concern, status="Pending preprocessing decision"):
    issue_rows.append({"Issue": issue, "Feature(s)": features, "Evidence": evidence,
        "Potential Concern": concern, "Decision Status": status})

for _, row in affected_missing.iterrows():
    register("Missing values", row["Feature"], f"{int(row['Missing count']):,} rows ({row['Missing percentage']:.3f}%)",
             "Meaning and availability of missing values need investigation.")
if duplicate_extra.any():
    register("Exact repeated rows", "All columns", f"{int(duplicate_extra.sum()):,} extra copies; {int(duplicate_involved.sum()):,} involved rows",
             "No obvious unique booking identifier; repeated rows may represent distinct bookings.")
for name, column, mask in validity_checks:
    if mask.any():
        register(name, column, f"{int(mask.sum()):,} rows", "Check source definitions and recording context.", "Requires contextual investigation")
if unexpected_hotel.any():
    register("Unexpected hotel category", "hotel", f"{int(unexpected_hotel.sum()):,} rows", "Confirm grouping labels.", "Requires contextual investigation")
for name, features, mask in [
    ("Zero total guests", "adults, children, babies", zero_guests),
    ("Zero total stay", "stays_in_weekend_nights, stays_in_week_nights", zero_stay),
    ("Negative ADR", "adr", negative_adr), ("Zero ADR", "adr", zero_adr)]:
    if mask.any():
        register(name, features, f"{int(mask.sum()):,} rows ({percent(mask.sum(), len(df)):.3f}%)",
                 "Recorded values need context before treatment.", "Requires contextual investigation")
for _, row in iqr_summary.loc[iqr_summary["Outside bounds"] > 0].iterrows():
    register("IQR screening candidates", row["Feature"], f"{int(row['Outside bounds']):,} rows ({row['Percentage of non-null']:.3f}% of non-null); IQR={row['IQR']:g}",
             "Screening flag is not proof of an invalid observation.", "Requires contextual investigation")
for _, row in category_summary.iterrows():
    for label in ["Empty strings", "Whitespace-only", "Leading/trailing whitespace", "Case-variant groups", "Context-label rows"]:
        if row[label] > 0:
            register(label, row["Feature"], f"{int(row[label]):,} {'groups' if label == 'Case-variant groups' else 'rows'}",
                     "Confirm category meanings in dataset documentation.", "Requires contextual investigation")
issue_register = pd.DataFrame(issue_rows, columns=["Issue", "Feature(s)", "Evidence", "Potential Concern", "Decision Status"])
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(issue_register)

,Issue,Feature(s),Evidence,Potential Concern,Decision Status
0,Missing values,company,"112,593 rows (94.307%)",Meaning and availability of missing values need investigation.,Pending preprocessing decision
1,Missing values,agent,"16,340 rows (13.686%)",Meaning and availability of missing values need investigation.,Pending preprocessing decision
2,Missing values,country,488 rows (0.409%),Meaning and availability of missing values need investigation.,Pending preprocessing decision
3,Missing values,children,4 rows (0.003%),Meaning and availability of missing values need investigation.,Pending preprocessing decision
4,Exact repeated rows,All columns,"31,994 extra copies; 40,165 involved rows",No obvious unique booking identifier; repeated rows may represent distinct bookings.,Pending preprocessing decision
5,Zero total guests,"adults, children, babies",180 rows (0.151%),Recorded values need context before treatment.,Requires contextual investigation
6,Zero total stay,"stays_in_weekend_nights, stays_in_week_nights",715 rows (0.599%),Recorded values need context before treatment.,Requires contextual investigation
7,Negative ADR,adr,1 rows (0.001%),Recorded values need context before treatment.,Requires contextual investigation
8,Zero ADR,adr,"1,959 rows (1.641%)",Recorded values need context before treatment.,Requires contextual investigation
9,IQR screening candidates,lead_time,"3,005 rows (2.517% of non-null); IQR=142",Screening flag is not proof of an invalid observation.,Requires contextual investigation


## 14 — Initial data-quality conclusions

The following Markdown findings are generated from this run's calculated tables. They separate observed evidence from pending decisions and remain reproducible if the input changes.

In [13]:
def markdown_table(table):
    # Format only presentation values; source data remains untouched.
    def text(value):
        if pd.isna(value):
            return "N/A"
        if isinstance(value, float):
            return f"{value:.3f}"
        return str(value)
    header = "| " + " | ".join(table.columns) + " |"
    divider = "| " + " | ".join("---" for _ in table.columns) + " |"
    rows = ["| " + " | ".join(text(value) for value in row) + " |" for row in table.itertuples(index=False, name=None)]
    return "\n".join([header, divider, *rows])

missing_text = "; ".join(f"{row['Feature']}: {int(row['Missing count']):,} ({row['Missing percentage']:.3f}%)"
    for _, row in affected_missing.iterrows()) or "No missing values detected"
duplicate_text = (f"{int(duplicate_extra.sum()):,} extra identical copies ({percent(duplicate_extra.sum(), len(df)):.3f}% of rows); "
    f"{int(duplicate_involved.sum()):,} rows belong to repeated groups ({percent(duplicate_involved.sum(), len(df)):.3f}%). "
    f"There are {len(row_frequencies):,} distinct full-row patterns, {len(repeated_groups):,} repeated groups, "
    f"and a maximum identical-row frequency of {int(row_frequencies.max()):,}. "
    "Duplicates have not been removed. Without an obvious unique booking identifier, identical records cannot automatically be treated as accidental duplicates.")
unusual_text = (f"Zero total guests: {int(zero_guests.sum()):,} ({percent(zero_guests.sum(), len(df)):.3f}%); "
    f"zero total stay: {int(zero_stay.sum()):,} ({percent(zero_stay.sum(), len(df)):.3f}%). "
    f"Guest totals are unevaluable for {int(total_guests.isna().sum()):,} records because at least one component is missing. "
    f"ADR ranges from {df['adr'].min():g} to {df['adr'].max():g}; {int(negative_adr.sum()):,} negative and {int(zero_adr.sum()):,} zero ADR records. "
    f"The {len(validity_checks)} basic validity checks produced {int(validity_summary['Flagged rows'].sum()):,} flags (counts may overlap). "
    f"Unexpected hotel-category rows: {int(unexpected_hotel.sum()):,}. "
    "These checks do not establish that every record is valid; unusual observations need context.")
iqr_text = (f"The 1.5 × IQR screen flagged {int(iqr_any.sum()):,} distinct records "
    f"({percent(iqr_any.sum(), len(df)):.3f}%) in at least one of {len(screening_columns)} selected measurements/counts. "
    f"{int(iqr_summary['IQR'].eq(0).sum())} screened fields have IQR = 0. "
    "An IQR flag is not evidence of an invalid record. Feature counts overlap, and percentages in the feature table use non-null counts. "
    "No observations were removed or capped; contextual investigation and later EDA are required.")
category_text = (f"Among {len(text_columns)} text columns, detected {int(category_summary['Empty strings'].sum()):,} empty-string cells, "
    f"{int(category_summary['Whitespace-only'].sum()):,} whitespace-only cells, "
    f"{int(category_summary['Leading/trailing whitespace'].sum()):,} cells with edge whitespace, and "
    f"{int(category_summary['Case-variant groups'].sum()):,} case-variant groups. "
    "These checks describe parsed values; default CSV parsing may represent empty fields and recognised missing tokens as nulls. "
    "Context labels below are not automatically invalid.")
hotel_text = ("The tables compare observed counts and within-hotel percentages. Differences are descriptive, "
    "do not rank data quality, do not establish causes, and do not demonstrate better performance from hotel-specific models.")
pending_text = ("- Investigate the meaning of missing values in the affected fields before choosing treatment.\n"
    "- Establish whether identical full-row patterns are distinct bookings before considering duplicate removal.\n"
    "- Clarify zero-guest, zero-stay, negative/zero ADR, and extreme-value records using context.\n"
    "- Review IQR candidates in later EDA, particularly fields with IQR = 0.\n"
    "- Verify detected context-label meanings in dataset documentation.\n"
    "- Consider the observed hotel-specific patterns when justifying preprocessing decisions.\n"
    "- Conduct the dedicated leakage audit separately; no features have been removed.")
conclusion_markdown = (
    "### 1. Missing data\n\n" + missing_text + ".\n\n"
    "### 2. Duplicate records\n\n" + duplicate_text + "\n\n"
    "### 3. Invalid/unusual observations\n\n" + unusual_text + "\n\n"
    "### 4. Outlier candidates\n\n" + iqr_text + "\n\n"
    "### 5. Categorical consistency\n\n" + category_text + "\n\n" + markdown_table(category_examples) + "\n\n"
    "### 6. City Hotel vs Resort Hotel quality differences\n\n" + hotel_text + "\n\n"
    + markdown_table(hotel_comparison) + "\n\n"
    "**No cleaning or preprocessing has been applied in this notebook. All treatment decisions will be justified during the preprocessing stage.**"
)
display(Markdown(conclusion_markdown))

# Build report text from the same results; this notebook does not write files.
report_markdown = (
    "# Data Quality Findings\n\n## Dataset\n\n"
    f"Source: `data/raw/hotel_bookings.csv`. Shape: **{df.shape[0]:,} rows × {df.shape[1]} columns**. "
    f"Calculated in `notebooks/02_data_quality_audit.ipynb` using pandas {pd.__version__}. "
    "Percentages use all rows unless labelled within-hotel or non-null. Audit only; no cleaning or preprocessing has been applied.\n\n"
    "## Missing Values\n\n" + missing_text + ". Missing does not automatically mean incorrect.\n\n"
    "## Duplicate Records\n\n" + duplicate_text + "\n\n"
    "## Unusual / Potentially Invalid Values\n\n" + unusual_text + "\n\n"
    "### Categorical consistency\n\n" + category_text + "\n\n" + markdown_table(category_examples) + "\n\n"
    "## Outlier Candidates\n\n" + iqr_text + "\n\n"
    + markdown_table(iqr_summary[["Feature", "IQR", "Outside bounds", "Percentage of non-null"]]) + "\n\n"
    "## City Hotel vs Resort Hotel Observations\n\n" + hotel_text + "\n\n"
    + markdown_table(hotel_comparison) + "\n\nFeature-specific missingness (denominator: records of that hotel):\n\n"
    + markdown_table(missing_by_hotel) + "\n\n"
    "## Decisions Pending for Preprocessing\n\n" + pending_text + "\n\n"
    "All treatment decisions will be justified during preprocessing. No rows, missing values, categories, or target values have been changed; no processed dataset, charts, or models were created.\n"
)

### 1. Missing data

company: 112,593 (94.307%); agent: 16,340 (13.686%); country: 488 (0.409%); children: 4 (0.003%).

### 2. Duplicate records

31,994 extra identical copies (26.798% of rows); 40,165 rows belong to repeated groups (33.642%). There are 87,396 distinct full-row patterns, 8,171 repeated groups, and a maximum identical-row frequency of 180. Duplicates have not been removed. Without an obvious unique booking identifier, identical records cannot automatically be treated as accidental duplicates.

### 3. Invalid/unusual observations

Zero total guests: 180 (0.151%); zero total stay: 715 (0.599%). Guest totals are unevaluable for 4 records because at least one component is missing. ADR ranges from -6.38 to 5400; 1 negative and 1,959 zero ADR records. The 16 basic validity checks produced 0 flags (counts may overlap). Unexpected hotel-category rows: 0. These checks do not establish that every record is valid; unusual observations need context.

### 4. Outlier candidates

The 1.5 × IQR screen flagged 64,004 distinct records (53.609%) in at least one of 13 selected measurements/counts. 8 screened fields have IQR = 0. An IQR flag is not evidence of an invalid record. Feature counts overlap, and percentages in the feature table use non-null counts. No observations were removed or capped; contextual investigation and later EDA are required.

### 5. Categorical consistency

Among 12 text columns, detected 0 empty-string cells, 0 whitespace-only cells, 0 cells with edge whitespace, and 0 case-variant groups. These checks describe parsed values; default CSV parsing may represent empty fields and recognised missing tokens as nulls. Context labels below are not automatically invalid.

| Feature | Observation | Example | Rows |
| --- | --- | --- | --- |
| meal | Context label | 'Undefined' | 1169 |
| market_segment | Context label | 'Undefined' | 2 |
| distribution_channel | Context label | 'Undefined' | 5 |

### 6. City Hotel vs Resort Hotel quality differences

The tables compare observed counts and within-hotel percentages. Differences are descriptive, do not rank data quality, do not establish causes, and do not demonstrate better performance from hotel-specific models.

| Hotel | Hotel records | Measure | Rows | Percentage within hotel |
| --- | --- | --- | --- | --- |
| City Hotel | 79330 | Any missing value | 79283 | 99.941 |
| City Hotel | 79330 | Duplicate copies | 25902 | 32.651 |
| City Hotel | 79330 | Duplicate involvement | 31748 | 40.020 |
| City Hotel | 79330 | Zero guests | 167 | 0.211 |
| City Hotel | 79330 | Zero stay | 331 | 0.417 |
| City Hotel | 79330 | Negative ADR | 0 | 0.000 |
| City Hotel | 79330 | Zero ADR | 1208 | 1.523 |
| Resort Hotel | 40060 | Any missing value | 39890 | 99.576 |
| Resort Hotel | 40060 | Duplicate copies | 6092 | 15.207 |
| Resort Hotel | 40060 | Duplicate involvement | 8417 | 21.011 |
| Resort Hotel | 40060 | Zero guests | 13 | 0.032 |
| Resort Hotel | 40060 | Zero stay | 384 | 0.959 |
| Resort Hotel | 40060 | Negative ADR | 1 | 0.002 |
| Resort Hotel | 40060 | Zero ADR | 751 | 1.875 |

**No cleaning or preprocessing has been applied in this notebook. All treatment decisions will be justified during the preprocessing stage.**

### Final integrity verification

Re-read the CSV solely to confirm that `df` still equals the raw parsed data, including row order, values, columns, and dtypes. Confirm the raw file's SHA-256 hash also remains unchanged.

In [14]:
assert df.shape == original_shape
assert df.columns.tolist() == original_columns
pd.testing.assert_frame_equal(df, pd.read_csv(data_path))
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == raw_hash_before
print("Verified: raw file hash unchanged; DataFrame rows, columns, values, and dtypes unchanged.")

Verified: raw file hash unchanged; DataFrame rows, columns, values, and dtypes unchanged.
